# otto-GPT(318M) 에이전트 — Colab (진짜 모델 + 안전망 + UI)

In [ ]:
!pip -q install sentencepiece gradio
import os
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR='/content/drive/MyDrive/otto_gpt'

DOCS = {'ai_bootcamp_notes.md': '# AI 부트캠프 커리큘럼 안내\n\n## 학습 목표\nAI 트랙은 딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다. 수료 후에는\n이미지 분류 모델과 RAG 기반 챗봇을 직접 만들 수 있는 수준을 목표로 합니다.\n\n## 주차별 내용\n1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과\nCNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다.\n5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.\n\n## RAG 실습 내용\nRAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에\n저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에\n넣고 LLM이 답변을 생성합니다.\n\n## 프로젝트 평가 기준\n최종 프로젝트는 동작 여부, 코드 구조, 그리고 발표로 평가됩니다. 단순히 개념을\n아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다.\n', 'startupcode_faq.md': '# 스타트업코드(Startupcode) 서비스 안내\n\n## 회사 소개\n스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는\n개발 교재 플랫폼 "어댑터즈(Adapterz)"와 온라인 코딩 부트캠프가 있습니다.\n2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.\n\n## 어댑터즈(Adapterz)란\n어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는\n서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다.\n교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.\n\n## 부트캠프 과정\n온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가\n있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션\n개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.\n\n## 수강료와 환불 정책\n부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후\n14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한\n잔여 금액의 일부만 환불됩니다.\n\n## 수료증과 취업 지원\n모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서\n첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.\n\n## 문의 방법\n서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr)\n또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.\n'}
os.makedirs('/content/docs', exist_ok=True)
for _n,_c in DOCS.items(): open(f'/content/docs/{_n}','w',encoding='utf-8').write(_c)
os.environ['RAG_DOCS_DIR']='/content/docs'
print('문서 저장 완료')

In [ ]:
from __future__ import annotations
# ===== model.py =====
"""otto-GPT 모델 정의 (02_otto_gpt 노트북과 동일). 추론용으로 04_agent 에 포함."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    block_size: int = 512     
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 624
    dropout: float = 0.1
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.n_head, self.n_embd, self.dropout = cfg.n_head, cfg.n_embd, cfg.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4*cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),   
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),   
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.transformer.wpe.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
            return logits, loss
        logits = self.lm_head(x[:, [-1], :])
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPT 모델 정의 완료")

# ===== parser.py =====
"""모델이 내뱉는 도구 호출 문자열을 파싱한다.

예) 'weather(도시="부산")' -> ('weather', {'도시': '부산'})
    'calculator(수식="12*7")' -> ('calculator', {'수식': '12*7'})

도구 호출이 아니면(일반 답변이면) None 을 반환한다.
"""

import re

# 도구명은 ASCII, 인자명은 한글 가능(\w 는 유니코드 매칭)
_CALL = re.compile(r'^\s*([A-Za-z_]\w*)\s*\((.*)\)\s*$', re.DOTALL)
_ARG = re.compile(r'(\w+)\s*=\s*"([^"]*)"')


def parse_tool_call(text: str) -> tuple[str, dict[str, str]] | None:
    """도구 호출이면 (이름, kwargs), 아니면 None."""
    if not text:
        return None
    m = _CALL.match(text.strip())
    if not m:
        return None
    name = m.group(1)
    kwargs = dict(_ARG.findall(m.group(2)))
    if not kwargs:  # 괄호는 있는데 key="val" 형식이 없으면 도구 호출로 보지 않음
        return None
    return name, kwargs

# ===== extract.py =====
"""에이전트 레이어 인자 재추출.

otto_gpt(57M)는 '어떤 도구를 쓸지'(라우팅)는 잘하나 '정확한 인자 복사'가 약하다
(예: 345+678 → 346+679, 오늘 환율 → 오늘 환율이, rag 인자 깨짐).
→ 모델의 라우팅 결정은 신뢰하되, 정밀한 인자는 사용자 원문에서 직접 추출한다.
"""

import re

# 단일 이항 산술식: 12*7, 345 + 678, 100/4 ...
_MATH = re.compile(r"(-?\d+(?:\.\d+)?)\s*([-+*/])\s*(-?\d+(?:\.\d+)?)")
# search 질의에서 떼어낼 꼬리말
_SEARCH_TAIL = re.compile(r"\s*(검색해줘|검색해|좀\s*찾아줘|찾아줘|검색|알려줘)\s*$")


def extract_args(tool: str, question: str, model_args: dict) -> dict:
    """도구별로 신뢰 가능한 인자를 만든다. 못 뽑으면 모델 인자로 폴백."""
    q = question.strip()

    if tool == "calculator":
        norm = q.replace("×", "*").replace("÷", "/")
        m = _MATH.search(norm)
        if m:
            return {"수식": f"{m.group(1)}{m.group(2)}{m.group(3)}"}
        return model_args

    if tool == "rag":
        # 모델 인자가 자주 깨짐 → 원문 질문을 그대로 (RAG는 자연어 질문 처리 OK)
        return {"질문": q}

    if tool == "search":
        cleaned = _SEARCH_TAIL.sub("", q).strip()
        return {"검색어": cleaned or model_args.get("검색어", "")}

    # weather 등은 모델 인자가 대체로 정확 → 그대로 사용
    return model_args

# ===== local_router.py =====
"""오프라인 룰 기반 라우터 (otto 없이도 에이전트가 돌도록).

otto_gpt 가 학습한 라우팅을 규칙으로 흉내낸다 — 어떤 도구를 쓸지만 결정하고,
정확한 인자는 agent 의 extract_args 가 원문에서 채운다(그래서 더미 인자 OK).
실제 모델을 쓰려면 OttoModel().generate_fn 을 Agent 에 주입.
"""

import re


_PROMPT_SPLIT = "### 지시:\n"


def local_router(prompt: str) -> str:
    q = prompt.split(_PROMPT_SPLIT, 1)[-1].split("\n", 1)[0].strip() if _PROMPT_SPLIT in prompt else prompt.strip()

    # 1) 산술 → calculator (extract 가 원문에서 수식 재추출)
    if _MATH.search(q.replace("×", "*").replace("÷", "/")):
        return 'calculator(수식="0")'

    # 2) 날씨 → weather (도시 추출)
    if any(w in q for w in ("날씨", "기온", "비 와")):
        m = re.search(r"([가-힣]+?)\s*(?:의)?\s*(?:날씨|기온)", q)
        return f'weather(도시="{m.group(1) if m else ""}")'

    # 3) 검색성 질의 → search (extract 가 꼬리말 제거해 채움)
    if any(w in q for w in ("검색", "찾아", "시세", "환율", "예매", "조회")):
        return 'search(검색어="")'

    # 4) 그 외(지식/사실 질문) → rag (extract 가 원문 질문으로 채움)
    return 'rag(질문="")'

# ===== hybrid.py =====
"""환각 차단 안전 라우터.

otto(57M)는 학습 분포 밖 입력에서 도구 호출 대신 자유 텍스트(환각)를 내뱉을 수 있다.
이 래퍼는 otto 출력이 **유효한 도구 호출일 때만** 사용하고, 아니면 규칙 라우터로 폴백한다.
→ 모델의 자유 생성 텍스트가 사용자 답변으로 절대 도달하지 않는다.
  (모든 답 = 도구 결과 / 문서 근거 / 정직한 '못 찾음')
"""

from typing import Callable



def make_safe_router(model_fn: Callable[[str], str]) -> Callable[[str], str]:
    """모델 라우터를 환각 안전 버전으로 감싼다."""

    def _route(prompt: str) -> str:
        out = (model_fn(prompt) or "").strip()
        if parse_tool_call(out):
            return out  # otto 가 제대로 라우팅 → 그대로 사용
        return local_router(prompt)  # 환각/실패 → 규칙으로 안전하게 도구 선택

    return _route

# ===== local_rag.py =====
"""오프라인 로컬 RAG 검색 (의존성 0, stdlib 만).

2단계: ① 질문에 맞는 **단락**을 TF-IDF 로 찾고 ② 그 단락 안에서 질문에 가장
맞는 **문장**만 골라 간결히 답한다. (단락 통째 반환 X)
Gemini/서버 없이도 in-domain 지식 질문에 근거 기반으로 답하기 위한 폴백.
"""

import glob
import math
import os
import re
from collections import Counter

try:
    _BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 노트북 셀에 인라인된 경우 __file__ 없음
    _BASE = os.getcwd()
_DEFAULT_DOCS = os.path.join(_BASE, "..", "03_rag", "langchain_rag", "docs")
DOCS_DIR = os.environ.get("RAG_DOCS_DIR", _DEFAULT_DOCS)


def _ngrams(text: str, n: int = 2) -> list[str]:
    """문자 n-gram. 한국어 조사(은/는/이/가...)를 흡수해 부분일치를 견고하게.
    예) '구독료'와 '구독료는' 이 'ㄱ구독','독료' n-gram 을 공유."""
    s = "".join(re.findall(r"[가-힣a-z0-9]", text.lower()))
    return [s[i : i + n] for i in range(len(s) - n + 1)]


def _sentences(paragraph: str) -> list[str]:
    """단락을 문장 단위로 분리. 헤딩(#) 줄 제외, 줄바꿈 이어붙인 뒤 마침표 기준 분리."""
    text = " ".join(
        ln.strip() for ln in paragraph.split("\n") if not ln.strip().startswith("#")
    )
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) >= 8]


class LocalRetriever:
    def __init__(self, docs_dir: str = DOCS_DIR):
        self.paras: list[tuple[str, str]] = []  # (source, paragraph)
        for path in sorted(glob.glob(os.path.join(docs_dir, "*.md"))):
            src = os.path.basename(path)
            with open(path, encoding="utf-8") as f:
                text = f.read()
            for para in re.split(r"\n\s*\n", text):
                if len(para.strip()) >= 20:
                    self.paras.append((src, para.strip()))
        # 문자 n-gram idf (코퍼스 전체 기준 — 희귀 표현일수록 변별력↑)
        n = max(len(self.paras), 1)
        df: Counter = Counter()
        for _, text in self.paras:
            for g in set(_ngrams(text)):
                df[g] += 1
        self.idf = {g: math.log((n + 1) / (df_g + 1)) + 1 for g, df_g in df.items()}

    def _score(self, query: str, text: str) -> float:
        qg = set(_ngrams(query))
        tg = set(_ngrams(text))
        return sum(self.idf.get(g, 1.0) for g in qg & tg)

    def best_paragraph(self, query: str) -> tuple[str, str] | None:
        best = max(self.paras, key=lambda p: self._score(query, p[1]), default=None)
        # 흔한 n-gram(어미 등)만 걸린 경우 배제: 의미 있는 매칭 점수 요구
        if best is None or self._score(query, best[1]) < 2.0:
            return None
        return best

    def answer(self, query: str) -> tuple[str, str] | None:
        """① 최적 단락 → ② 그 안에서 질문을 가장 잘 담은 문장 1~2개."""
        hit = self.best_paragraph(query)
        if hit is None:
            return None
        src, para = hit
        sents = _sentences(para) or [para]
        # 단락 제목에 든 단어 = 주제어 → 문장 선택 시 약화(질문의 '의도'어가 이기게)
        head_grams = set(
            _ngrams(" ".join(l for l in para.split("\n") if l.strip().startswith("#")))
        )
        qg = set(_ngrams(query))

        def sscore(s: str) -> float:
            sg = set(_ngrams(s))
            return sum(
                self.idf.get(g, 1.0) * (0.3 if g in head_grams else 1.0)
                for g in qg & sg
            )

        ranked = sorted(sents, key=sscore, reverse=True)
        top_score = sscore(ranked[0])
        # 기본 1문장. 2번째는 점수가 '거의 동률'(>=0.9)일 때만 — 한 세트로 묶인 답
        picked = [s for s in ranked if sscore(s) >= top_score * 0.9][:2]
        return src, " ".join(picked)


_retriever: LocalRetriever | None = None


def local_rag(질문: str = "", 검색어: str = "", **_) -> str:
    """로컬 문서에서 질문에 맞는 문장을 찾아 간결히 반환. (offline, no quota)"""
    global _retriever
    q = (질문 or 검색어).strip()
    if not q:
        return "질문이 비었습니다."
    if _retriever is None:
        _retriever = LocalRetriever()
    res = _retriever.answer(q)
    if res is None:
        return "제공된 문서에서 답을 찾을 수 없습니다."
    src, text = res
    return f"{text} (출처: {src}, 로컬검색)"

# ===== tools.py =====
"""도구 레지스트리 + 실행.

calculator 는 실제로 계산하고(안전한 산술 eval), weather/search 는 외부 키가 없으므로
목업(mock) 결과를 돌려준다. 실제 API 키가 있으면 해당 함수만 교체하면 된다.
"""

import ast
import json
import operator
import os
import urllib.error
import urllib.request

# ── calculator: 안전한 산술 평가 (eval 미사용) ──
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("허용되지 않은 수식")


def calculator(수식: str = "", **_) -> str:
    try:
        result = _safe_eval(ast.parse(수식, mode="eval"))
        return f"{수식} = {result}"
    except Exception:
        return f"계산할 수 없는 수식: {수식!r}"


def weather(도시: str = "", **_) -> str:
    # 목업: 실제로는 기상 API 호출. 도시명으로 결정적 가짜 데이터 생성.
    temp = 18 + (len(도시) * 3) % 12
    return f"{도시}의 현재 날씨: 맑음, 기온 {temp}도 (mock)"


def search(검색어: str = "", **_) -> str:
    # 목업: 실제로는 검색 API 호출.
    return f"'{검색어}' 검색 결과 상위 항목 (mock): 관련 문서 3건"


# ── RAG 도구: 03_rag 의 FastAPI /query 를 HTTP 로 호출 (venv 분리 유지) ──
# 지식이 필요한 질문은 작은 라우터(otto_gpt) 대신 RAG 가 근거 기반으로 답한다.
RAG_API = os.environ.get("RAG_API", "http://127.0.0.1:8000/query")


def rag_search(검색어: str = "", 질문: str = "", **_) -> str:
    """03_rag 파이프라인에 질문을 던져 근거 기반 답변 + 출처를 받는다.
    otto_gpt 는 search(검색어=...) 로 호출하므로 검색어/질문 둘 다 받는다."""
    q = (질문 or 검색어).strip()
    if not q:
        return "검색어가 비었습니다."
    try:
        req = urllib.request.Request(
            RAG_API,
            data=json.dumps({"question": q}).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            d = json.loads(resp.read().decode("utf-8"))
        srcs = ", ".join(d.get("sources", []))
        return f"{d.get('answer', '')}" + (f" (출처: {srcs})" if srcs else "")
    except urllib.error.URLError:
        # 서버/Gemini 없을 때 → 오프라인 로컬 문서 검색으로 폴백
        try:
            return local_rag(질문=q)
        except Exception:
            return "[RAG 서버 연결 실패 + 로컬 문서 없음]"
    except Exception as e:
        return f"[RAG 오류: {e}]"


# 기본 레지스트리 (search = 목업, rag = 실제 03_rag 호출)
REGISTRY = {
    "calculator": calculator,
    "weather": weather,
    "search": search,
    "rag": rag_search,
}


def rag_registry() -> dict:
    """search 를 실제 RAG 로 바인딩한 레지스트리.
    otto_gpt 가 이미 search 로 라우팅하므로 재학습 없이 지식 질문을 RAG 로 보낸다."""
    return {**REGISTRY, "search": rag_search, "rag": rag_search}

# ===== agent.py =====
"""에이전트 실행 루프.

모델 = 라우터(도구+인자 결정), 에이전트 = 파싱 -> 도구 실행 -> 응답.

흐름:
  질문 -> [모델] -> 출력
    ├─ 도구 호출이면 -> 파싱 -> 도구 실행 -> 관찰(observation) -> 응답
    └─ 아니면 -> 그 자체가 직접 답변

모델은 `generate_fn(prompt: str) -> str` 형태로 주입한다(otto_gpt, mock, 무엇이든).
이렇게 하면 모델 없이도 루프를 검증할 수 있다.
"""

from typing import Callable


# otto_gpt 멀티태스크 튜닝과 동일한 프롬프트 포맷
PROMPT = "### 지시:\n{q}\n\n### 응답:\n"


class Agent:
    def __init__(self, generate_fn: Callable[[str], str], tools: dict = REGISTRY):
        self.generate_fn = generate_fn
        self.tools = tools

    def run(self, question: str) -> dict:
        raw = self.generate_fn(PROMPT.format(q=question)).strip()
        parsed = parse_tool_call(raw)

        # 1) 도구 호출이 아니면 → 직접 답변
        if parsed is None:
            return {"type": "direct", "raw": raw, "answer": raw}

        name, model_args = parsed

        # 2) 모르는 도구
        if name not in self.tools:
            return {
                "type": "unknown_tool", "raw": raw, "tool": name, "args": model_args,
                "answer": f"알 수 없는 도구를 호출했습니다: {name}",
            }

        # 3) 인자 재추출: 모델의 라우팅은 신뢰, 정밀 인자는 원문에서 보정
        args = extract_args(name, question, model_args)

        # 4) 도구 실행 → 관찰
        observation = self.tools[name](**args)
        return {
            "type": "tool", "raw": raw, "tool": name,
            "model_args": model_args,   # 모델이 낸 (보정 전) 인자
            "args": args,               # 실제 실행에 쓴 (보정 후) 인자
            "observation": observation,
            "answer": self._synthesize(question, observation),
        }

    def _synthesize(self, question: str, observation: str) -> str:
        """도구 결과를 자연어 답변으로. (현재는 템플릿 — 모델이 관찰→답변을
        학습하지 않았으므로. 추후 'observation -> answer' 데이터로 학습하면
        여기서 self.generate_fn 으로 2차 합성하도록 교체)."""
        return observation

# ===== otto_model.py =====
"""otto_gpt_instruct 추론 래퍼 → Agent 에 넣을 generate_fn 제공.

Colab(또는 체크포인트가 있는 로컬)에서:
    from agent import Agent
    gen = OttoModel("/content/drive/MyDrive/otto_gpt").generate_fn
    agent = Agent(generate_fn=gen)
    print(agent.run("부산 날씨 어때?"))
"""

import os

import torch
import torch.nn.functional as F
import sentencepiece as spm



class OttoModel:
    # 318M instruct 우선, 없으면 57M instruct 폴백
    _PREFERRED = ["otto_gpt_350m_instruct.pt", "otto_gpt_instruct.pt"]

    def __init__(self, project_dir: str, ckpt_name: str | None = None):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sp = spm.SentencePieceProcessor(
            model_file=f"{project_dir}/tokenizer_ko.model"
        )
        if ckpt_name is None:  # 가장 좋은 체크포인트 자동 선택 (318M → 57M)
            ckpt_name = next(
                (c for c in self._PREFERRED if os.path.exists(f"{project_dir}/ckpt/{c}")),
                self._PREFERRED[-1],
            )
        print(f"[OttoModel] 라우터 체크포인트: {ckpt_name}")
        ckpt = torch.load(f"{project_dir}/ckpt/{ckpt_name}", map_location=self.device)
        self.config = GPTConfig(**ckpt["config"])
        self.model = GPT(self.config).to(self.device)
        self.model.load_state_dict(ckpt["model"])
        self.model.eval()

    @torch.no_grad()
    def generate_fn(self, prompt: str, max_new_tokens: int = 150,
                    temperature: float = 0.7, top_k: int = 40,
                    rep_penalty: float = 1.3) -> str:
        bs = self.config.block_size
        x = torch.tensor([self.sp.encode(prompt)], dtype=torch.long, device=self.device)
        start = x.size(1)
        for _ in range(max_new_tokens):
            logits, _ = self.model(x[:, -bs:])
            logits = logits[:, -1, :] / temperature
            for t in set(x[0].tolist()):
                logits[0, t] /= rep_penalty
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            if nxt.item() == self.sp.eos_id():
                break
            x = torch.cat((x, nxt), dim=1)
        # 프롬프트 이후 생성분만 반환 (Agent 가 도구호출/답변을 파싱)
        return self.sp.decode(x[0, start:].tolist()).strip()

print('에이전트 코드 정의 완료')

In [ ]:
# 진짜 otto(318M instruct 자동선택) + 환각 안전망
otto = OttoModel(PROJECT_DIR)
agent = Agent(generate_fn=make_safe_router(otto.generate_fn))
print('에이전트 준비')

for q in ['12*7 계산해줘','345+678 얼마야?','어댑터즈 구독료 얼마야?','부산 날씨 어때?','인공지능이 뭐야?','건강하게 사는 방법 알려줘']:
    r=agent.run(q); print('Q:',q); print('  otto원출력:',repr(r['raw'][:40]))
    if r['type']=='tool':
        if r.get('model_args')!=r.get('args'): print('  인자보정:',r['model_args'],'→',r['args'])
        print('  도구',r['tool'],'→',r['observation'][:60])
    print('  ▶',r['answer'][:70],'\n')

In [ ]:
import gradio as gr
def chat(question):
    r=agent.run(question)
    d=(f"🔧 {r['tool']} {r.get('args')}\notto원출력: {r['raw'][:60]}" if r['type']=='tool' else f"otto: {r['raw'][:80]}")
    return r['answer'], d
gr.Interface(fn=chat, inputs=gr.Textbox(label='질문'),
    outputs=[gr.Textbox(label='답변(환각 없음)'),gr.Textbox(label='동작 상세')],
    title='otto-GPT(318M) 에이전트',
    examples=[['12*7 계산해줘'],['어댑터즈 구독료 얼마야?'],['부산 날씨 어때?'],['인공지능이 뭐야?']],
).launch(share=True)